In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mabs_funcs as mf

import matplotlib.colors as colors
import matplotlib.cm as cm

In [ ]:
Uavg = mf.define_rings()
Uavg_phy = mf.du_phy * Uavg
rdata, rcount = mf.CalCR(Uavg)

In [ ]:

file_loc_v1='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/v1'
file_loc_v2='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/v2'


In [ ]:
r1,r2,r3=13,13,13
print('correspong ell=%f'%(2*np.pi*Uavg_phy[r1]))
NRi1 = range(1,2) 
# NRi1 = [i for i in range(1, 301) if i != 181 and i!=182]
NRi2 = range(1,2) 

mabs_R = []
for i in NRi1:
    filename = "%s/bsout_final_ra%d_rb%d_rc%d_R%d" % (file_loc_v1, r1, r2, r3, i)
    Ntri1, mabs1, numc1 = mf.read_bsout_float(filename)
    mabs_F1 = mf.fold_quadrants(mabs1)
    mabs_R.append(mabs_F1)

mabs_R_2 = []
for i in NRi2:
    filename = "%s/bsout_final2_ra%d_rb%d_rc%d_R%d" % (file_loc_v2, r1, r2, r3, i)
    Ntri2, mabs2, numc2 = mf.read_bsout_float_v2(filename)
    mabs_F2 = mf.fold_quadrants(mabs2)
    mabs_R_2.append(mabs_F2)
    

In [ ]:
plt.imshow(mabs1/numc1)

In [ ]:
# np.max(mabs),mabs[mf.nc10-1:mf.nc10+1,mf.nc10-1:mf.nc10+2]

## Data Prep

In [ ]:
numc_mean_1 = mf.fold_quadrants(numc1)
numc_mean_2 = mf.fold_quadrants(numc2)

mabs_mean_1 = mabs_F1
mabs_mean_2 = mabs_F2


mabs_T_1 = mabs_mean_1/mf.mabs_norm/numc_mean_1
mabs_T_2 = mabs_mean_2/mf.mabs_norm/numc_mean_2
# mabs_final = np.nan_to_num(mabs_final, nan=0)

# mabs_stdT_F = np.std(mabs_R,axis=0)/mf.mabs_norm/numc_mean_F/np.sqrt(len(NRi1)-1)
# mabs_stdT_NF = np.std(mabs_R_2,axis=0)/mf.mabs_norm/numc_mean_NF/np.sqrt(len(NRi2)-1)

In [ ]:
np.nanstd(mabs_T_1/mabs_T_2)

In [ ]:
# Nnc_hf = mf.nc10//2+1
Nnc  = 2*(mf.nc10+1 - mf.n_nancut)-1 
Nnc_hf = Nnc//2+1

mabs_final_1=  mf.cut_nan(mabs_T_1,mf.n_nancut)  #mabs_T[:Nnc_hf,Nnc_hf-1:Nnc_hf+mf.nc10]
mabs_final_2=  mf.cut_nan(mabs_T_2,mf.n_nancut)

# mabs_final_stdT_F = mf.cut_nan(mabs_stdT_F,mf.n_nancut)
# mabs_final_stdT_NF= mf.cut_nan(mabs_stdT_NF,mf.n_nancut)
plt.imshow(mabs_final_1,origin='lower',extent=[-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa,0,Nnc_hf*mf.dnu_mwa],cmap='coolwarm') #,vmin=0,vmax=0.03
plt.colorbar(location='top')

In [ ]:
np.shape(mabs_final_1)

In [ ]:
plt.imshow(mabs_final_2,origin='lower',extent=[-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa,0,Nnc_hf*mf.dnu_mwa],cmap='coolwarm') #,vmin=0,vmax=0.03
plt.colorbar(location='top')

# From DFT Theory

In [ ]:
Nnc = mabs_final_1.shape[1] #  mf.nc10//2 + 1

L  = Nnc * mf.dnu_mwa * mf.rc_prime

print(L)

indices = np.fft.fftshift(np.fft.fftfreq(Nnc, d=1.0/Nnc))

ell1, ell2, ell3 = 2.*np.pi*Uavg_phy[r1], 2.*np.pi*Uavg_phy[r2], 2.*np.pi*Uavg_phy[r3]

i_vals, j_vals = np.meshgrid(indices, indices, indexing='ij')
data_k = mf.Bk_func(ell1, ell2, ell3, i_vals, j_vals, L, 1.0)

In [ ]:
j_vals[0]

In [ ]:
(2*np.pi/L)*93

In [ ]:
xx = mf.Bk2Bdnu_fft(data_k)
plt.imshow(xx,origin='lower',extent=[-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa,0,Nnc_hf*mf.dnu_mwa],cmap='coolwarm')
plt.colorbar(location='top')

In [ ]:
def inset_lp(ax1,data, est_key, est_data, est_sd, text):
    lp_xvals = np.linspace(xmin,xmax,np.size(data))
    ax1_inset = inset_axes(ax1, width="90%", height="40%", loc="upper center")
    ax1_inset.plot( lp_xvals, data, 'k-', lw=1.3)
    ax1_inset.set_xticks([])  # Remove x-ticks for clarity
    ax1_inset.set_yticks([])  # Remove y-ticks for clarity
    # ax1_inset.set_frame_on(True)  # Keep the frame visible
    ax1_inset.set_facecolor((1, 1, 1, 0.6))  # Make the background transparent
    ax1_inset.text(0.07, 0.9, text, transform=ax1.transAxes, fontsize=15, color="k", ha="left", va="top")
    # ax1_inset.spines['top'].set_visible(False)  # Hide frame (optional)
    # ax1_inset.spines['right'].set_visible(False)
    # ax1_inset.spines['left'].set_visible(False)
    # ax1_inset.spines['bottom'].set_visible(False)
    if est_key==1:
        # ax1_inset.errorbar(lp_xvals, est_data, yerr=est_sd, fmt='.', capsize=1, color='g', markersize=.05)
        ax1_inset.errorbar(lp_xvals, est_data, yerr=0, fmt='.', capsize=1, color='g', markersize=.07)
        ax1_inset.fill_between(lp_xvals, est_data - est_sd, est_data + est_sd, color='orange', alpha=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

vmin = min( np.min(mabs_final_1), np.min(mabs_final_2))
vmax = max( np.max(mabs_final_1), np.max(mabs_final_2))
xmin = -Nnc_hf*mf.dnu_mwa
xmax = Nnc_hf*mf.dnu_mwa
ymin =  0
ymax = Nnc_hf*mf.dnu_mwa
cmap = 'coolwarm'
# Create a figure with two subplots side by side
fig, (ax1) = plt.subplots(1, 1, figsize=(8, 8),sharey=True)


# Plot the second image with common normalization
im1 = ax1.imshow(mabs_final_1, origin='lower',
                 extent=[xmin, xmax, ymin, ymax],
                 cmap=cmap, vmin=vmin, vmax=vmax)
ax1.set_title(r"Estimate without flagging")


# Create a ScalarMappable with the common normalization and colormap
norm = colors.Normalize(vmin=vmin, vmax=vmax)
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Manually create an axes for the colorbar.
# The list defines [left, bottom, width, height] in figure coordinates (0 to 1).
cbar_ax = fig.add_axes([0.055, .67, .929, 0.03])  # right,up,length,width,
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{1},\Delta\nu_{2})$',fontsize=12,labelpad=15)
cbar.ax.tick_params(labelsize=10)


ax1.set_xlabel(r'$\Delta\nu_{1}$',fontsize=13)
ax1.set_ylabel(r'$\Delta\nu_{2}$',fontsize=13)


ax1.tick_params(labelsize=10)

# ax1.set_xscale('symlog', linthresh=.1)  # change linthresh as needed
# ax1.set_yscale('symlog', linthresh=.1)  # or 'log' if you want log y-axis and y > 0

#### Lineplots ###################
inset_lp(ax1,mabs_final_1[0],0,0,0,'')
inset_lp(ax1,mabs_final_1[1],0,0,0,'')
inset_lp(ax1,mabs_final_1[10],0,0,0,'')


# Adjust layout to accommodate the colorbar manually
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.subplots_adjust(wspace=.1, hspace=0)
# plt.savefig('4_fbe/Plots/mabs.pdf', format='pdf', dpi=500, bbox_inches='tight')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Example shape
Ny, Nx = mabs_final_1.shape

# Create edges for pcolormesh
x_edges = np.linspace(xmin, xmax, Nx + 1)
y_edges = np.linspace(ymin, ymax, Ny + 1)  # start at 0.1 if you want to allow log scale

fig, ax = plt.subplots(figsize=(8, 4))  # Wider and flatter

mesh = ax.pcolormesh(x_edges, y_edges, mabs_final_1, cmap=cmap,vmin=vmin, vmax=vmax, shading='auto')

# mesh = ax.pcolormesh(x_edges, y_edges, mabs_final_1,
#                      cmap=cmap,
#                      norm=LogNorm(vmin=vmin, vmax=vmax),
#                      shading='auto')

# Set symlog for x-axis, and optionally log or linear for y
ax.set_xscale('symlog', linthresh=.1)  # change linthresh as needed
ax.set_yscale('symlog', linthresh=.1)  # or 'log' if you want log y-axis and y > 0

# Labels, colorbar, etc.
fig.colorbar(mesh, ax=ax)
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
plt.tight_layout()
plt.show()


In [ ]:
i=1
plt.plot(mabs_final_1[i],'.')


In [ ]:
i=1
plt.plot(mabs_final_1[i],'.')

In [ ]:
lp_xvals = np.linspace(xmin,xmax,np.size(xx[0]))

In [ ]:
# np.savez("mabs_plot",xx,mabs_final_NF,mabs_final_stdT_NF,mabs_final_F, mabs_final_stdT_F,lp_xvals)

# Inverse FFT -> $\Delta\nu$ To $k_\parallel$ space

In [ ]:
Nnc = mabs_final_1.shape[1] #  mf.nc10//2 + 1
L   = Nnc * mf.dnu_mwa * mf.rc_prime
rc4 = mf.rc**4
Lsq = L**2
dkll = 2*np.pi/L

In [ ]:
res_1 = mf.Bdnu2Bk_fft(mabs_final_1)
res_2 = mf.Bdnu2Bk_fft(mabs_final_2)

In [ ]:
plt.imshow(np.log10(res_1))
plt.colorbar()
plt.xlim(50,80)
plt.ylim(50,80)

In [ ]:
plt.imshow(np.log10(res_2))
plt.colorbar()
plt.xlim(50,80)
plt.ylim(50,80)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm

vmin = min(np.min(res_2), np.min(res_1))
vmax = max(np.max(res_2), np.max(res_1))

# Create a figure with two subplots side by side
fig, (ax1) = plt.subplots(1, 1, figsize=(8, 8))


# Plot the second image with common normalization
im1 = ax1.imshow((res_1), origin='lower',
                 extent=[-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1), -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1)],
                 cmap='coolwarm', vmin=vmin, vmax=vmax)
ax1.set_title(r"Estimated")

# Create a ScalarMappable with the common normalization and colormap
norm = colors.Normalize(vmin=vmin, vmax=vmax)
sm = cm.ScalarMappable(norm=norm, cmap='coolwarm')
sm.set_array([])

# Manually create an axes for the colorbar.
# The list defines [left, bottom, width, height] in figure coordinates (0 to 1).
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])  # right,up,length,width,
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

ax1.set_xlabel(r'$k_{\parallel3}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

ax1.set_xlim(-.5,.5)
ax1.set_ylim(-.5,.5)
ax2.set_ylim(-.5,.5)
# Adjust layout to accommodate the colorbar manually
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm

vmin = min(np.min(res_2), np.min(res_1))
vmax = max(np.max(res_2), np.max(res_1))

fig, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# Plot the image
im1 = ax1.imshow(res_1, origin='lower',
                 extent=[-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1)],
                 cmap='coolwarm', vmin=vmin, vmax=vmax)

ax1.set_title(r"Estimated")

# Add symlog scale to both axes
ax1.set_xscale('symlog', linthresh=0.001)
ax1.set_yscale('symlog', linthresh=0.001)

# Colorbar setup
norm = colors.Normalize(vmin=vmin, vmax=vmax)
sm = cm.ScalarMappable(norm=norm, cmap='coolwarm')
sm.set_array([])

# Manual colorbar axes
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

# Axis labels and limits
ax1.set_xlabel(r'$k_{\parallel1}$')
ax1.set_ylabel(r'$k_{\parallel2}$')
ax1.set_xlim(-0.1, 0.1)
ax1.set_ylim(-0.1, 0.1)

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm

# Set vmin and vmax for color normalization
vmin = min(np.min(res_2), np.min(res_1))
vmax = max(np.max(res_2), np.max(res_1))

# Create grid edges for pcolormesh (one more than data size)
Ny, Nx = res_1.shape
x_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nx + 1)
y_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Ny + 1)

# Create the figure and axis
fig, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# Use LogNorm or Normalize depending on your data range
norm = colors.Normalize(vmin=vmin, vmax=vmax)

# Plot with pcolormesh
mesh = ax1.pcolormesh(x_edges, y_edges, res_1,
                      cmap='coolwarm', norm=norm, shading='auto')

# Apply symlog to x and y axes
ax1.set_xscale('symlog', linthresh=0.001)
ax1.set_yscale('symlog', linthresh=0.001)

# Title and labels
ax1.set_title(r"Estimated")
ax1.set_xlabel(r'$k_{\parallel3}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

# Set axis limits
ax1.set_xlim(-0.1, 0.1)
ax1.set_ylim(-0.1, 0.1)

# Colorbar (manual positioning)
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])  # [left, bottom, width, height]
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

# Adjust layout to leave space for colorbar
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.colors import SymLogNorm

# Prepare vmin, vmax, and linthresh for SymLogNorm
linthresh = 1e5
linscale = 1.0
vmax =  np.max(res_1)
vmin = -vmax

# Grid for pcolormesh
Ny, Nx = res_1.shape
x_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nx + 1)
y_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Ny + 1)

# Create figure and axis
fig, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# Use SymLogNorm for symlog color normalization
norm = SymLogNorm(linthresh=linthresh, linscale=linscale, vmin=vmin, vmax=vmax)

# Plot using pcolormesh
mesh = ax1.pcolormesh(x_edges, y_edges, res_1,
                      cmap='coolwarm', norm=norm, shading='auto')

# Set symlog scales for x and y axes
ax1.set_xscale('symlog', linthresh=0.01)
ax1.set_yscale('symlog', linthresh=0.01)

# Axis labels and title
ax1.set_title(r"Estimated")
ax1.set_xlabel(r'$k_{\parallel1}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

# Axis limits
ax1.set_xlim(-0.5, 0.5)
ax1.set_ylim(-0.5, 0.5)

# Add horizontal colorbar
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.colors import SymLogNorm
from matplotlib.colors import LogNorm

# Prepare vmin, vmax, and linthresh for SymLogNorm
linthresh = 1e5
linscale = 1.0
vmax =  np.max(np.abs(res_1))
vmin = np.min(np.abs(res_1))

# Grid for pcolormesh
Ny, Nx = res_1.shape
x_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nx + 1)
y_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Ny + 1)

# Create figure and axis
fig, ax1 = plt.subplots(1, 1, figsize=(6, 6))

# Use SymLogNorm for symlog color normalization
norm = LogNorm(vmin=vmin, vmax=vmax)#SymLogNorm(linthresh=linthresh, linscale=linscale, vmin=vmin, vmax=vmax)

# Plot using pcolormesh
mesh = ax1.pcolormesh(x_edges, y_edges, res_1,
                      cmap='coolwarm', norm=norm, shading='auto')

# Set symlog scales for x and y axes
# ax1.set_xscale('symlog', linthresh=0.01)
# ax1.set_yscale('symlog', linthresh=0.01)

# ax1.set_xscale('log')
# ax1.set_yscale('log')

# Axis labels and title
ax1.set_title(r"Estimated")
ax1.set_xlabel(r'$k_{\parallel1}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

# Axis limits
ax1.set_xlim(-0.2, 0.2)
ax1.set_ylim(-0.2, 0.2)

# Add horizontal colorbar
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.colors import SymLogNorm
from matplotlib.colors import LogNorm

# Prepare vmin, vmax, and linthresh for SymLogNorm
linthresh = 1e5
linscale = 1.0
vmax =  np.max(np.abs(res_1))
vmin = np.min(np.abs(res_1))

# Grid for pcolormesh
Ny, Nx = res_1.shape
x_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nx + 1)
y_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Ny + 1)

# Create figure and axis
fig, ax1 = plt.subplots(1, 1, figsize=(6, 6))

# Use SymLogNorm for symlog color normalization
norm = LogNorm(vmin=vmin, vmax=vmax)#SymLogNorm(linthresh=linthresh, linscale=linscale, vmin=vmin, vmax=vmax)

# Plot using pcolormesh
mesh = ax1.pcolormesh(x_edges, y_edges, res_1,
                      cmap='coolwarm', norm=norm, shading='auto')

# Set symlog scales for x and y axes
# ax1.set_xscale('symlog', linthresh=0.01)
# ax1.set_yscale('symlog', linthresh=0.01)

# ax1.set_xscale('log')
# ax1.set_yscale('log')

# Axis labels and title
ax1.set_title(r"Estimated")
ax1.set_xlabel(r'$k_{\parallel1}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

# Axis limits
ax1.set_xlim(-0.2, 0.2)
ax1.set_ylim(-0.2, 0.2)

# Add horizontal colorbar
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
i=np.shape(res_1)[0]//2-10

xrange = np.linspace(-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1),Nnc )
plt.plot(xrange,res_1[i],'.')
plt.xlim(0.01,1)
plt.xscale('log')
plt.yscale('log')

In [ ]:
dkll*(Nnc_hf-1)

In [ ]:
kp = 2*np.pi*Uavg_phy[r1]/mf.rc
wedge = (mf.rc/mf.rc_prime/154.255)*kp

In [ ]:
kp

In [ ]:
i=np.shape(res_1)[0]//2-10

xrange = np.linspace(-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1),Nnc )
plt.plot(xrange,res_1[i],'.')
plt.vlines([wedge],1e14,1e17)
plt.xlim(0.01,1)
# plt.xscale('log')
plt.yscale('log')

In [ ]:
751//2

In [ ]:
-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1)

# Bin the data

In [ ]:
from scipy.stats import binned_statistic

rowi = np.shape(res_1)[0]//2

res_arr = res_1[rowi]
n_bins = 20

# Define bin edges based on xrange
bin_edges = np.linspace(xrange.min(), xrange.max(), n_bins + 1)

# Compute the mean of 'res' in each bin
res_bin_means, _, _ = binned_statistic(xrange, res_arr, statistic='mean', bins=bin_edges)

# Compute the bin-average of 'xrange'
xrange_bin_means, _, _ = binned_statistic(xrange, xrange, statistic='mean', bins=bin_edges)

# print("Binned res averages:", res_bin_means)
# print("Binned xrange averages:", xrange_bin_means)

plt.plot(xrange_bin_means,res_bin_means,'-o')

### Manual

In [ ]:
# res_arr = res[65]

# # Define the number of bins
# n_bins = 15

# # Create bin edges spanning the range of xrange
# bin_edges = np.linspace(xrange.min(), xrange.max(), n_bins + 1)

# # Prepare lists to store binned averages
# binned_res = []
# bin_avg_xrange = []

# # Loop over each bin and compute averages
# for i in range(n_bins):
#     # For the last bin, include the right edge
#     if i == n_bins - 1:
#         mask = (xrange >= bin_edges[i]) & (xrange <= bin_edges[i+1])
#     else:
#         mask = (xrange >= bin_edges[i]) & (xrange < bin_edges[i+1])
    
#     # Check if any values fall into the current bin
#     if np.sum(mask) > 0:
#         binned_res.append(np.mean(res_arr[mask]))
#         bin_avg_xrange.append(np.mean(xrange[mask]))
#     else:
#         # If the bin is empty, you can decide how to handle it; here we use NaN.
#         binned_res.append(np.nan)
#         bin_avg_xrange.append(np.nan)

# # Print the binned averages
# print("Binned res averages:", binned_res)
# print("Binned xrange averages:", bin_avg_xrange)

# plt.plot(bin_avg_xrange,binned_res,'o')

# Cylinderical BS for many realizations


In [ ]:
from scipy.stats import binned_statistic_2d

Nnc    =  2*(mf.nc10+1 - mf.n_nancut)-1  ##cylbs_mean_F.shape[1] 
Nnc_hf = Nnc//2+1
L      = Nnc * mf.dnu_mwa * mf.rc_prime
rc4    = mf.rc**4
Lsq    = L**2
dkll   = 2*np.pi/L

In [ ]:
dkll

In [ ]:
def bin_2D_cylbs(cylbs_mean_NF,n_bins):
    x = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), cylbs_mean_NF.shape[0])  #np.linspace(-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1),Nnc )
    y = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), cylbs_mean_NF.shape[1])
    X, Y = np.meshgrid(x, y)
    
    # Flatten the arrays to feed into the binned statistic function
    x_flat = X.ravel()
    y_flat = Y.ravel()
    data_flat_NF = cylbs_mean_NF.ravel()
    
    # Bin the data into 30x30 bins using the mean as the statistic
    binned_clbs_NF, x_edges, y_edges, binnumber = binned_statistic_2d( x_flat, y_flat, data_flat_NF, statistic='mean', bins=n_bins)

    return binned_clbs_NF



In [ ]:
r1,r2,r3=13,13,13
print('correspong ell=%f'%(2*np.pi*Uavg_phy[r1]))
NRi1 = range(1,2) 
NRi2 = range(1,2) 
n_bins=30
cylbs_R = []
for i in NRi1:
    filename = "%s/bsout_final_ra%d_rb%d_rc%d_R%d" % (file_loc_v1, r1, r2, r3, i)
    Ntri1, mabs1, numc1 = mf.read_bsout_float(filename)
    mabs_F1 = mf.fold_quadrants(mabs1)
    numc_F1 = mf.fold_quadrants(numc1)
    mabs_T_F = mabs_F1/mf.mabs_norm/numc_F1
    mabs_final_F=  mf.cut_nan(mabs_T_F,mf.n_nancut)
    res_F = mf.Bdnu2Bk_fft(mabs_final_F)
    binned_res_F = bin_2D_cylbs(res_F,n_bins)
    cylbs_R.append(binned_res_F)

# cylbs_R_2 = []
# for i in NRi2:
#     filename = "%s/bsout_final2_ra%d_rb%d_rc%d_R%d" % (file_loc_v2, r1, r2, r3, i)
#     Ntri2, mabs2, numc2 = mf.read_bsout_float(filename)
#     mabs_F2 = mf.fold_quadrants(mabs2)
#     numc_F2 = mf.fold_quadrants(numc2)
#     mabs_T_NF = mabs_F2/mf.mabs_norm/numc_F2
#     mabs_final_NF=  mf.cut_nan(mabs_T_NF,mf.n_nancut)
#     res_NF = mf.Bdnu2Bk_fft(mabs_final_NF)
#     binned_res_NF = bin_2D_cylbs(res_NF,n_bins)
#     cylbs_R_2.append(binned_res_NF)

In [ ]:

cylbs_mean_F = cylbs_R[0]




In [ ]:
# Nnc  = 2*(mf.nc10+1 - mf.n_nancut)-1 
# Nnc_hf = Nnc//2+1

In [ ]:
# Nnc = cylbs_mean_F.shape[1] #  mf.nc10//2 + 1
# L   = Nnc * mf.dnu_sim * mf.rc_prime
# rc4 = mf.rc**4
# Lsq = L**2
# dkll = 2*np.pi/L

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm

vmin = np.min(cylbs_mean_F)
vmax = np.max(cylbs_mean_F)

# Create a figure with two subplots side by side
fig, (ax1) = plt.subplots(1, 1, figsize=(8, 8))



# Plot the second image with common normalization
im1 = ax1.imshow((cylbs_mean_F), origin='lower',
                 extent=[-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1), -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1)],
                 cmap='coolwarm', vmin=vmin, vmax=vmax)
ax1.set_title(r"Estimated")

# Create a ScalarMappable with the common normalization and colormap
norm = colors.Normalize(vmin=vmin, vmax=vmax)
sm = cm.ScalarMappable(norm=norm, cmap='coolwarm')
sm.set_array([])

# Manually create an axes for the colorbar.
# The list defines [left, bottom, width, height] in figure coordinates (0 to 1).
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])  # right,up,length,width,
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')


ax1.set_xlabel(r'$k_{\parallel3}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

ax1.set_xlim(-.5,.5)
ax1.set_ylim(-.5,.5)

# Adjust layout to accommodate the colorbar manually
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
dkll*(Nnc_hf-1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.colors import SymLogNorm

cmap = 'viridis_r'
# Prepare vmin, vmax, and linthresh for SymLogNorm
linthresh = 1e10
linscale = 1e0
vmax =  np.max(cylbs_mean_F)
vmin = np.min(cylbs_mean_F)

# Grid for pcolormesh
Ny, Nx = cylbs_mean_F.shape
x_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nx + 1)
y_edges = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Ny + 1)

# Create figure and axis
fig, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# Use SymLogNorm for symlog color normalization
norm = SymLogNorm(linthresh=linthresh, linscale=linscale, vmin=vmin, vmax=vmax)

# Plot using pcolormesh
mesh = ax1.pcolormesh(x_edges, y_edges, cylbs_mean_F,
                      cmap=cmap, norm=norm, shading='auto')

# Set symlog scales for x and y axes
ax1.set_xscale('symlog', linthresh=1)
ax1.set_yscale('symlog', linthresh=1)

# Axis labels and title
ax1.set_title(r"Estimated")
ax1.set_xlabel(r'$k_{\parallel1}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

# Axis limits
# ax1.set_xlim(-0.5, 0.5)
# ax1.set_ylim(-0.5, 0.5)

# Add horizontal colorbar
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
i=17

xrange = np.linspace(-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1),Nnc )
plt.plot(cylbs_mean_F[i],'.-')

In [ ]:
import numpy as np
from scipy.stats import binned_statistic_2d

n_bins = 20

x = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), data_k.shape[0])  #np.linspace(-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1),Nnc )
y = np.linspace(-dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), data_k.shape[1])
X, Y = np.meshgrid(x, y)

# Flatten the arrays to feed into the binned statistic function
x_flat = X.ravel()
y_flat = Y.ravel()
# data_flat_NF = cylbs_mean_NF.ravel()
data_k_flat = data_k.ravel()

# Bin the data into 30x30 bins using the mean as the statistic
# binned_clbs_NF, x_edges, y_edges, binnumber = binned_statistic_2d( x_flat, y_flat, data_flat_NF, statistic='mean', bins=n_bins)
binned_datak, _, _, _ = binned_statistic_2d( x_flat, y_flat, data_k_flat, statistic='mean', bins=n_bins)

binned_clbs_NF=cylbs_mean_F

In [ ]:
_, x_edges, y_edges, binnumber = binned_statistic_2d( x_flat, y_flat, data_k_flat, statistic='mean', bins=n_bins)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])

In [ ]:
x_centers

In [ ]:

# np.savez("cylbs_plot",binned_datak,cylbs_mean_NF,cylbs_stdT_NF,cylbs_mean_F, cylbs_stdT_F,lp_xvals)


# 3D BS

In [ ]:
file_loc_v1='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/v1'
file_loc_v2='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/v2'

In [ ]:
k1_bins = 12
t_bins  = 5
mu_bins = 5


n_shapes = 669#365
Nnc  = 2*(mf.nc10+1 - mf.n_nancut)-1   ##mf.nc10+1
LL   = mf.dnu_mwa * mf.rc_prime
L    = Nnc * LL
print(mf.dnu_mwa,LL,L)
dkll = 2 * np.pi / L

indices = np.fft.fftshift(np.fft.fftfreq(Nnc, d=1.0/Nnc))
i_vals, j_vals = np.meshgrid(indices, indices, indexing='ij')
k2_para = dkll * i_vals
k3_para = dkll * j_vals
k1_para = - k3_para - k2_para

k1_edges = np.logspace(np.log10(0.002), np.log10(4.624), k1_bins + 1)
t_edges  = np.linspace(.5, 1., t_bins + 1)
mu_edges = np.linspace(.5, 1., mu_bins + 1)
bin_edges = [k1_edges, t_edges, mu_edges]

# file_loc='/scratch/sukhdeep/project_MABS/3VC_B/4_fbe/results/final'
# file_loc='/scratch/sukhdeep/project_MABS/3VC_B/4_fbe/results/flagged/coll'

In [ ]:
dkll

In [ ]:
Rlzn = 1
# if Rlzn<=300:
#     add_loc = '/below15'
# else:
#     add_loc= '/below15_set2_61to100'

dnu_file_loc = file_loc_v1 #+ add_loc

In [ ]:
bsA,numA = mf.get_Ana_3DBS( bin_edges,n_shapes, rdata, Uavg_phy, k2_para, k3_para, dnu_file_loc, Rlzn)

In [ ]:
# np.savez("ananonFlagColl_bs_num_12X5X5_v2",bsA,numA)

In [ ]:
i = 7
bispec = bsA[i]/numA[i]

In [ ]:
plt.imshow(bispec,origin='lower')


# Run

In [ ]:
dnu_file_loc = file_loc_v1 #file_loc_NF

bs,num = mf.get_binned_3DBS(bin_edges, n_shapes, rdata, Uavg_phy, k2_para, k3_para, dnu_file_loc, Rlzn)


# dnu_file_loc = file_loc + '/below15_set2_61to100'
# for Rlzn in range(301,501):
#     print(Rlzn)
#     bs,num = mf.get_binned_3DBS(bin_edges, n_shapes, rdata, Uavg_phy, k2_para, k3_para, dnu_file_loc, Rlzn)
#     bs_R.append(bs)

In [ ]:
# bs_mean = bs


In [ ]:
# np.savez("bs_mwa_data_12X5X5",bs,num)

In [ ]:
# bs_std_npz = np.load("bs_std_15X10X10.npz",allow_pickle=True)
# bs_m = bs_std_npz['arr_0']  
# bs_s = bs_std_npz['arr_1'] 

In [ ]:
np.allclose(bs_std,bs_s)

# Fast (not much)

In [ ]:
all_samples = mf.get_samples(n_shapes, rdata, Uavg_phy, k2_para, k3_para)

In [ ]:
bs_R = []
dnu_file_loc = file_loc + '/below15'
for Rlzn in range(1,301):
    print(Rlzn)
    bs,num = mf.get_binned_3DBS_fast(bin_edges, rdata, n_shapes, all_samples, dnu_file_loc, Rlzn)
    bs_R.append(bs)


dnu_file_loc = file_loc + '/below15_set2_61to100'
for Rlzn in range(301,501):
    print(Rlzn)
    bs,num = mf.get_binned_3DBS_fast(bin_edges, rdata, n_shapes, all_samples, dnu_file_loc, Rlzn)
    bs_R.append(bs)

In [ ]:
bs_mean = np.mean(bs_R,axis=0)

tot_NRi = 500
bs_std = np.std(bs_R,axis=0)/np.sqrt(tot_NRi-1)

# Plots

In [ ]:
Rlzn = 1
if Rlzn <= 300:
    add_loc = '/below15'
else:
    add_loc= '/below15_set2_61to100'

dnu_file_loc = file_loc + add_loc


bsA,numA = mf.get_Ana_3DBS( bin_edges,n_shapes, rdata, Uavg_phy, k2_para, k3_para, dnu_file_loc, Rlzn)

In [ ]:
bs_std_npz = np.load("bs_std_15X10X10.npz",allow_pickle=True)
bs_m = bs_std_npz['arr_0']  
bs_s = bs_std_npz['arr_1'] 

In [ ]:
i = 6
Ana_bispec = bsA[i]/numA[i]
Est_bispec = bs_m[i]/numA[i]

plt.imshow(Ana_bispec,origin='lower')


In [ ]:
plt.imshow(Est_bispec,origin='lower')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm

cmap = 'viridis'

vmin = min(np.nanmin(Est_bispec), np.nanmin(Ana_bispec))
vmax = max(np.nanmax(Est_bispec), np.nanmax(Ana_bispec))

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Plot the first image with common normalization
im1 = ax1.imshow((Ana_bispec), origin='lower',
                 extent=[0.5,1,0.5,1],
                 cmap=cmap, vmin=vmin, vmax=vmax)
ax1.set_title(r"Analytical")

# Plot the second image with common normalization
im2 = ax2.imshow((Est_bispec), origin='lower',
                 extent=[0.5,1,0.5,1],
                 cmap=cmap, vmin=vmin, vmax=vmax)
ax2.set_title(r"Estimated")

# Create a ScalarMappable with the common normalization and colormap
norm = colors.Normalize(vmin=vmin, vmax=vmax)
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Manually create an axes for the colorbar.
# The list defines [left, bottom, width, height] in figure coordinates (0 to 1).
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])  # right,up,length,width,
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

ax2.set_xlabel(r'$k_{\parallel3}$')
ax1.set_xlabel(r'$k_{\parallel3}$')
ax1.set_ylabel(r'$k_{\parallel2}$')
# Adjust layout to accommodate the colorbar manually
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
Est_bispec

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm

cmap = 'viridis'
Est_bispec[Est_bispec<0] = np.nan
# Ensure vmin is positive for log normalization.
raw_vmin = min(np.nanmin(Est_bispec), np.nanmin(Ana_bispec))
vmin = raw_vmin if raw_vmin > 0 else 1e-10  # adjust if necessary
vmax = max(np.nanmax(Est_bispec), np.nanmax(Ana_bispec))

# Create a log normalization instance.
log_norm = colors.LogNorm(vmin=vmin, vmax=vmax)

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Plot the first image with log normalization
im1 = ax1.imshow(Ana_bispec, origin='lower',
                 extent=[0.5, 1, 0.5, 1],
                 cmap=cmap, norm=log_norm)
ax1.set_title(r"Analytical")

# Plot the second image with log normalization
im2 = ax2.imshow(Est_bispec, origin='lower',
                 extent=[0.5, 1, 0.5, 1],
                 cmap=cmap, norm=log_norm)
ax2.set_title(r"Estimated")

# Create a ScalarMappable with the log normalization and colormap
sm = cm.ScalarMappable(norm=log_norm, cmap=cmap)
sm.set_array([])

# Manually create an axes for the colorbar.
# The list defines [left, bottom, width, height] in figure coordinates (0 to 1).
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(\ell_1,\ell_2,\ell_3,\Delta\nu_{21},\Delta\nu_{31})$')

ax2.set_xlabel(r'$k_{\parallel3}$')
ax1.set_xlabel(r'$k_{\parallel3}$')
ax1.set_ylabel(r'$k_{\parallel2}$')

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()


In [ ]:
raw_vmin = min(np.nanmin(Est_bispec), np.nanmin(Ana_bispec))
vmin = raw_vmin if raw_vmin > 0 else 1e-10  # adjust if necessary
vmax = max(np.nanmax(Est_bispec), np.nanmax(Ana_bispec))

In [ ]:
np.nanmin(Est_bispec), np.nanmin(Ana_bispec)

In [ ]:
Est_bispec[Est_bispec<0] = np.nan

In [ ]:
mf.dnu_sim